# 06 · Advanced RAG III: Post-retrieval
**Concepts:** (A) reranking: retrieve wide, keep the best; (B) contextual compression: keep only the sentences that answer the question.
**TAC scenario:** skim twenty hits, shortlist four, highlight the lines that matter.

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMListwiseRerank, LLMChainExtractor, EmbeddingsFilter
from rag_common import golden_set, doc_ids, hit, reciprocal_rank, show, get_embeddings

wide = vectorstore.as_retriever(search_kwargs={"k": 20})     # recall: cast a wide net
top4 = vectorstore.as_retriever(search_kwargs={"k": 4})      # baseline

## A. Reranking with an LLM (no extra service or key)

In [ ]:
reranker = LLMListwiseRerank.from_llm(llm=model, top_n=4)
reranked = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=wide)

q = "Scanners lose Wi-Fi whenever staff walk between floors"
print("Top-4 by vector similarity:"); show(top4.invoke(q))
print("\nTop-4 after reranking 20 candidates:"); show(reranked.invoke(q))

In [ ]:
sample = [g for g in golden_set() if g["type"] in ("single_hop", "multi_hop", "how_to")][:10]
for name, r in [("vector top-4", top4), ("rerank 20→4", reranked)]:
    ids = [doc_ids(r.invoke(g["question"])) for g in sample]
    h = sum(hit(i, g["expected_doc_ids"]) for i, g in zip(ids, sample))
    mrr = sum(reciprocal_rank(i, g["expected_doc_ids"]) for i, g in zip(ids, sample)) / len(sample)
    print(f"{name:14s} hit@4 {h}/{len(sample)}   MRR {mrr:.2f}")

## B. Contextual compression: extract only the relevant sentences

In [ ]:
extractor = LLMChainExtractor.from_llm(model)
compressed = ContextualCompressionRetriever(base_compressor=extractor, base_retriever=top4)

q = "What BFD timers should be used for provider BGP sessions on the AR-900?"
before, after = top4.invoke(q), compressed.invoke(q)
print(f"Before: {len(before)} chunks, {sum(len(d.page_content) for d in before)} chars")
print(f"After : {len(after)} chunks, {sum(len(d.page_content) for d in after)} chars\n")
for d in after:
    print(f"[{d.metadata['doc_id']}] {d.page_content}\n")

### A cheaper filter: drop chunks below an embedding-similarity threshold (no LLM call)

In [ ]:
emb_filter = ContextualCompressionRetriever(
    base_compressor=EmbeddingsFilter(embeddings=get_embeddings(), similarity_threshold=0.45),
    base_retriever=wide,
)
kept = emb_filter.invoke(q)
print(f"{len(kept)} of 20 candidates kept:", doc_ids(kept))

---
**🔧 TAC takeaway:** Reranking is a sharper Select; compression is literally Compress. Fewer, better tokens mean better answers at lower cost.